### Develop a Machine Translation system to translate public information content between English and any Indian language.

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

In [10]:
model_name = "facebook/nllb-200-distilled-600M"

# load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [11]:
# set source language
tokenizer.src_lang = "eng_Latn"

def translate_english_to_hindi(text):

    inputs = tokenizer(text, return_tensors="pt")

    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("hin_Deva"),
        max_length=100
    )

    translated_text = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)

    return translated_text[0]

In [12]:

text = "The train will arrive at platform number five at 10 AM."

print("English:", text)

translation = translate_english_to_hindi(text)

print("Hindi Translation:", translation)

English: The train will arrive at platform number five at 10 AM.
Hindi Translation: ट्रेन सुबह 10 बजे प्लेटफॉर्म नंबर पांच पर पहुंच जाएगी।


In [13]:
sentences = [
    "The hospital is open twenty four hours.",
    "Wear a mask in crowded areas.",
    "The bus will depart at 6 PM.",
    "Please maintain social distance."
]

for sentence in sentences:

    translated = translate_english_to_hindi(sentence)

    print("English:", sentence)
    print("Hindi:", translated)
    print()

English: The hospital is open twenty four hours.
Hindi: अस्पताल चौबीस घंटे खुला रहता है।

English: Wear a mask in crowded areas.
Hindi: भीड़ वाले स्थानों पर मास्क पहनें।

English: The bus will depart at 6 PM.
Hindi: बस शाम 6 बजे रवाना होगी।

English: Please maintain social distance.
Hindi: कृपया सामाजिक दूरी बनाए रखें।



In [14]:
data = [
    "The railway station is closed today.",
    "Please stand in the queue.",
    "The vaccination center opens at nine in the morning.",
    "Emergency services are available here."
]

translations = []

for sentence in data:
    translations.append(translate_english_to_hindi(sentence))

import pandas as pd

df = pd.DataFrame({
    "English": data,
    "Hindi": translations
})

df

,English,Hindi
0,The railway station is closed today.,रेलवे स्टेशन आज बंद है।
1,Please stand in the queue.,कृपया कतार में खड़े हो जाओ।
2,The vaccination center opens at nine in the mo...,टीकाकरण केंद्र सुबह नौ बजे खुलता है।
3,Emergency services are available here.,आपातकालीन सेवाएं यहां उपलब्ध हैं।


In [15]:
df.to_csv("english_hindi_translation.csv", index=False)

In [16]:
from nltk.translate.bleu_score import sentence_bleu

reference = [["ट्रेन", "सुबह", "10", "बजे", "प्लेटफॉर्म", "पांच", "पर", "पहुंचेगी"]]
candidate = translation.split()

score = sentence_bleu(reference, candidate)

print("BLEU Score:", score)

BLEU Score: 0.4518010018049224
